### scRNA-seq Tutorial 2: Reference mapping

In this tutorial, you will learn how to annotate the cells in your dataset through transfering the labels from an annotated embryonic reference. We will be mapping our datasets onto post-conception week 3, 4 and 5 cells from [PMID 37192616](https://pubmed.ncbi.nlm.nih.gov/37192616/).

First, we import the packages we need to access the functions we use. If you need to use another package, install it and import it below. The main package we use to map single cell RNA-seq data is [scvi](https://docs.scvi-tools.org/en/stable/), which we install in the next cell.

In [ ]:
#%pip install scvi-tools

In [3]:
import os
import sys
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scvi

c:\Users\rafae\OneDrive\Documents\Uni\PhD\code\BOO2026_scRNAseq_scripts\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load the datasets you wish to use for annotation and mapping. In this example, we use the concatenated dataset from the first tutorial as our 'query', and we load an already pre-processed dataset of embryonic cells from post-conception weeks (PCW) 3, 4 and 5. 


In [4]:
#Load the data you wish to annotate; In this example, we use the concatenated dataset from the first tutorial
adata = sc.read_h5ad ("data/3802619/GSE167011_IPS_lung_differentiation_concat.h5ad")
adata

AnnData object with n_obs × n_vars = 1290 × 3604
    obs: 'n_genes_by_counts', 'total_counts', 'pct_counts_mit', 'pct_counts_ribo', 'day'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection', 'mean', 'std'
    uns: 'day_colors', 'hvg', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [ ]:
reference = sc.read_10x_mtx('data/references/PCW3_outs/filtered_feature_bc_matrix')
reference

AnnData object with n_obs × n_vars = 18854 × 38606
    var: 'gene_ids', 'feature_types'

Let's first take a look at out reference dataset and check out all the cell types that have been annotated. The annotations can be found in the ```cell_type``` column.

In [ ]:
#UMAP code
sc.pl.umap(
    reference,
    color=['cell_type'],
    legend_loc="best",
    use_raw = True
)

In order to perform the label transfer from the reference, we will train a model that projects the gene expression data from our reference into latent space. Then we map the query data into the same latent space using this trained model. Finally, we assign labels to query cells based on their proximity to labeled reference cells in this space, typically using a classifier such as k-nearest neighbors or a built-in classifier (as in scANVI).

First, we must concatenate the two datasets in one object so that the model can learn a shared space. To do this, we have to subset the datasets to only common genes. 

In [ ]:
#Get common genes
common_genes = reference.var_names.intersection(adata.var_names)

#Subset to common genes
reference = reference[:, common_genes].copy()
adata = adata[:, common_genes].copy()

Now we combine the datasets into one object. We add a variable called ```batch```, which we can use to separate ```ref``` from ```query```. We further label the cells in the query as 'Unknown'.

In [ ]:
adata_combined = reference.concatenate(adata, batch_key="batch", batch_categories=["ref", "query"])
adata_combined.obs["cell_type"] = adata_combined.obs["cell_type"].fillna("Unknown")


Now we train the model on the reference

In [ ]:
#This step might take a while

scvi.model.SCANVI.setup_anndata(
    adata_combined,
    batch_key="batch",
    labels_key="cell_type"
)

model = scvi.model.SCANVI(adata_combined,
                        unlabeled_category="Unknown",
                        n_layers=2,
                        encode_covariates=True,
                        deeply_inject_covariates=True,
                        use_layer_norm="both",
                        use_batch_norm="none",
)
model.train(max_epochs=50)


Let's extract the coordinates of our cells in latent space.

In [ ]:
adata_combined.obsm["X_scVI"] = model.get_latent_representation()

Now we can predict the labels of our query data.

In [ ]:
query_mask = adata_combined.obs["batch"] == "query"
adata_combined.obs["cell_type"] = model.predict(adata_combined)[query_mask]
adata_combined.obs["cell_type"].head()

Let's visualise the mapping in latent space. We will separate the query by day to show where they have been mapped relative to reference, and separately show the cell type annotations.

In [ ]:
#Compute UMAP
sc.pp.neighbors(adata_combined, use_rep="X_scVI")
sc.tl.umap(adata_combined)

#Annotate the cells by reference, day5 or day 10
adata_combined.obs["mapping_view"] = "reference"

day5_mask = query_mask & (adata_combined.obs["day"].astype(str) == "day 5")
day10_mask = query_mask & (adata_combined.obs["day"].astype(str) == "day 10")

adata_combined.obs.loc[day5_mask, "mapping_view"] = "query_day5"
adata_combined.obs.loc[day10_mask, "mapping_view"] = "query_day10"

adata_combined.obs["mapping_view"] = pd.Categorical(
    adata_combined.obs["mapping_view"],
    categories=["reference", "query_day5", "query_day10"]
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --------------------------
# UMAP 1: mapping view
# --------------------------
sc.pl.umap(
    adata_combined,
    color="mapping_view",
    palette={
        "reference": "lightgrey",
        "query_day5": "red",
        "query_day10": "blue",
    },
    ax=axes[0],
    show=False,
    title="Query mapped onto reference"
)

# --------------------------
# UMAP 2: combined labels
# --------------------------
sc.pl.umap(
    adata_combined,
    color="cell_type",
    ax=axes[1],
    show=False,
    title="Cell types (ref + predicted)"
)

plt.tight_layout()
plt.show()